# MOOSE inference (nb2): converted NIfTI → segmentations  —  model-specific

Runs `moosez` clinical-CT models on the GPU VM. Consumes the **Boundary-A** archive
`converted_nifti.tar.lz4` from nb1 and emits the **Boundary-B** archive
`segmentations.tar.lz4` with the canonical layout:
```
<SeriesInstanceUID>/<model>/segmentations/<multilabel>.nii.gz
<SeriesInstanceUID>/<model>/label_map.json      # {label_id: label_name}
```
nb3 (shared) joins each `label_name` against the model's SNOMED mapping to build
DICOM-SEG + pyradiomics + SR. This notebook is the *only* MOOSE-specific piece.

## Imports

In [ ]:
import json
import os
import shutil
import subprocess
import sys
import time
import traceback
from pathlib import Path

NOTEBOOK_START = time.time()
def _elapsed(s=None):
    return f"{time.time() - (s if s is not None else NOTEBOOK_START):.1f}s"
print(f"[T+{_elapsed()}] Imports complete")

## Parameters

In [ ]:
# Boundary-A archive produced by nb1 (local file on the same VM).
converted_nifti_path = "converted_nifti.tar.lz4"

# Short model identifier used in the Boundary-B layout (<uid>/<model>/...).
model_name = "moose"

# 'cuda' for GPU, 'cpu' for CPU-only (falls back to cpu if no GPU).
accelerator = "cuda"

# Reserved for checkpoint/resume on preemption (not yet wired in this notebook).
checkpoint_gcs = ""

# Model-specific knob injected via `papermill -f inference_params.yaml`.
moose_models = "clin_ct_organs,clin_ct_ribs,clin_ct_vertebrae"

## Extract Boundary-A archive

In [ ]:
models = [m.strip() for m in moose_models.split(',') if m.strip()]
NIFTI_DIR = Path('/tmp/converted_nifti')
SEG_DIR = Path('/tmp/segmentations')
for _d in (NIFTI_DIR, SEG_DIR):
    if _d.exists():
        shutil.rmtree(_d)
    _d.mkdir(parents=True, exist_ok=True)

subprocess.run(f'lz4 -d -c {converted_nifti_path} | tar -xf - -C {NIFTI_DIR.parent}',
               shell=True, check=True)
# nb1 packs the directory 'converted_nifti/<uid>/<uid>.nii.gz'; collapse if nested.
if (NIFTI_DIR / 'converted_nifti').is_dir():
    NIFTI_DIR = NIFTI_DIR / 'converted_nifti'
series_uids = sorted(d.name for d in NIFTI_DIR.iterdir() if d.is_dir())
print(f'MOOSE models : {models}')
print(f'Series       : {len(series_uids)}')

## GPU availability check (fall back to CPU if none)

In [ ]:
usage_metrics = {'series': {}, 'gpu': []}
try:
    import torch
    print(f'PyTorch {torch.__version__}  CUDA available: {torch.cuda.is_available()}')
    if not torch.cuda.is_available() and accelerator == 'cuda':
        print('WARNING: accelerator=cuda but no GPU found; falling back to cpu')
        accelerator = 'cpu'
except Exception as exc:
    print(f'PyTorch unavailable: {exc}')
    if accelerator == 'cuda':
        accelerator = 'cpu'
print(f'[T+{_elapsed()}] Accelerator = {accelerator}')

## Run moosez → Boundary-B layout

`moosez` writes `<out>/<uid>/moosez-<model>-<ts>/segmentations/<multilabel>.nii.gz`.
Each produced multilabel volume is moved to the canonical
`<uid>/<model>/segmentations/` and its `organ_indices` (moosez's own, authoritative
`{label_id: label_name}`) is written next to it as `label_map.json`.

In [ ]:
print(f'[T+{_elapsed()}] Importing moosez ...', flush=True)
from moosez import moose
moose_errors = []

for uid in series_uids:
    nii = NIFTI_DIR / uid / f'{uid}.nii.gz'
    if not nii.exists():
        cands = list((NIFTI_DIR / uid).glob('*.nii.gz'))
        if not cands:
            moose_errors.append(f'{uid}: no NIfTI found')
            continue
        nii = cands[0]
    work = Path('/tmp/moose_work') / uid
    if work.exists():
        shutil.rmtree(work)
    work.mkdir(parents=True, exist_ok=True)
    print(f'[T+{_elapsed()}] Series {uid}')
    series_times = {}
    for model in models:
        t0 = time.time()
        try:
            seg_paths, model_objs = moose(str(nii), [model], str(work), accelerator)
            series_times[model] = round(time.time() - t0, 1)
            if not seg_paths or not model_objs:
                moose_errors.append(f'{uid}/{model}: moose produced no output')
                continue
            dest = SEG_DIR / uid / model / 'segmentations'
            dest.mkdir(parents=True, exist_ok=True)
            for sp in seg_paths:
                shutil.copy(str(sp), str(dest / Path(sp).name))
            label_map = {str(k): v for k, v in dict(model_objs[0].organ_indices).items()}
            (SEG_DIR / uid / model / 'label_map.json').write_text(
                json.dumps({'model': model, 'labels': label_map}, indent=2))
            print(f'  {model}: {series_times[model]}s  ({len(label_map)} labels)')
        except Exception as exc:
            moose_errors.append(f'{uid}/{model}: {traceback.format_exc()}')
            print(f'  ERROR {model}: {exc}')
    usage_metrics['series'][uid] = {'moose_models_s': series_times}

if moose_errors:
    Path('inference_errors.txt').write_text('\n'.join(moose_errors))
print(f'[T+{_elapsed()}] Inference complete ({len(moose_errors)} error(s))')

## Package Boundary-B archive + usage metrics

In [ ]:
import csv
produced = [d for d in SEG_DIR.iterdir() if d.is_dir()]
if not produced:
    raise RuntimeError('No segmentations produced — see inference_errors.txt')

subprocess.run(f'tar -cf - -C {SEG_DIR.parent} {SEG_DIR.name} | lz4 > segmentations.tar.lz4',
               shell=True, check=True)
size_mb = Path('segmentations.tar.lz4').stat().st_size / (1024 ** 2)

usage_metrics['total_elapsed_s'] = round(time.time() - NOTEBOOK_START, 1)
with open('inference_UsageMetrics.csv', 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['SeriesInstanceUID', 'model', 'model_inference_s', 'run_total_elapsed_s'])
    for uid, m in usage_metrics['series'].items():
        for model, secs in m.get('moose_models_s', {}).items():
            w.writerow([uid, model, secs, usage_metrics['total_elapsed_s']])

print(f'[T+{_elapsed()}] Wrote segmentations.tar.lz4 ({size_mb:.1f} MB, {len(produced)} series)')